# E7. Robustez por grupo de captura
Usa las predicciones finales ya guardadas y el umbral congelado. Un grupo con una sola clase permite medir errores de clasificación, pero no AUROC.

In [ ]:
from pathlib import Path
from IPython import get_ipython

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()
)


def load_notebook(name):
    path = ROOT / "notebooks" / "shared" / (name + ".ipynb")
    get_ipython().run_line_magic("run", f'-i "{path.as_posix()}"')


load_notebook("workspace")

In [ ]:
load_notebook("experiment")
load_notebook("metrics")
frame, dataset_hash = cap_rows()
suite = RUNS.parent / "final" / dataset_hash
test = frame[frame.split == "test"].copy()
test["path"] = test.path.astype(str)
results = []
for name in json.loads((suite / "plan.json").read_text())["runs"]:
    run = RUNS / name
    threshold = json.loads((run / "threshold.json").read_text())["value"]
    predictions = pd.read_csv(suite / name / "predictions.csv").merge(
        test[["path", "group_id"]], on="path", validate="one_to_one"
    )
    for group, subset in predictions.groupby("group_id"):
        chosen = subset.score >= threshold
        results.append(
            dict(
                run=name,
                group_id=group,
                images=len(subset),
                normal=int((subset.label == 0).sum()),
                defect=int(subset.label.sum()),
                false_positives=int((chosen & (subset.label == 0)).sum()),
                false_negatives=int((~chosen & (subset.label == 1)).sum()),
                auroc=roc_auc_score(subset.label, subset.score)
                if subset.label.nunique() == 2
                else None,
            )
        )
report = pd.DataFrame(results)
report.to_csv(suite / "robustness.csv", index=False)
export_study_file(suite, "robustness.csv")
display(report)